<a href="https://colab.research.google.com/github/Nahla-Nabil/agentic-distillation-benchmark/blob/master/notebooks/02_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Training: all three conditions

Runs `base`, `sft_only`, and `distilled` through the same script
(`src/adbench/training/train.py`) — this notebook only orchestrates; all
the actual logic (config resolution, the per-step KD+SFT loss, data
formatting, the training loop) lives in `adbench.training.train` and
`adbench.training.losses`, and is unit-tested locally (`tests/test_train.py`,
`tests/test_losses.py`) without needing a GPU.

Requires `01_data_prep.ipynb` to have been run first (needs
`data/splits/train.jsonl`), and `00_setup_colab.ipynb` for the GPU deps.

In [1]:
# Private repo: create a GitHub personal access token (repo scope) and
# add it as a Colab secret named GH_TOKEN (key icon in the left sidebar,
# then enable notebook access for it) before running this cell.
# Skip this cell if this notebook's runtime already has the repo cloned
# (e.g. you ran 00_setup_colab.ipynb in this same session).
import os

if not os.path.isdir('/content/agentic-distillation-benchmark'):
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
    os.environ['GH_TOKEN'] = token
    !git clone https://$GH_TOKEN@github.com/Nahla-Nabil/agentic-distillation-benchmark.git

%cd /content/agentic-distillation-benchmark
!pip install -q -r requirements-colab.txt
!pip install -q -e .

/content/agentic-distillation-benchmark
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for adbench (pyproject.toml) ... done


In [2]:
from adbench.training.train import load_experiment_config, train_condition

experiment_config = load_experiment_config("configs/experiment.yaml")
models_config = load_experiment_config("configs/models.yaml")

## Condition 1 — base (no training)

Loads the student, attaches a freshly-initialized (untrained) LoRA adapter,
saves immediately — see `train.py::save_checkpoint()`'s docstring for why
this condition still goes through the same load+save path as the other two
rather than being special-cased in evaluation.

In [3]:
base_summary = train_condition("base", experiment_config, models_config)
print(base_summary)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.4: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Unsloth 2026.9.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.
Unsloth: Restored added_tokens_decoder metadata in /content/agentic-distillation-benchmark/checkpoints/base/tokenizer_config.json.


{'condition': 'base', 'checkpoint_dir': '/content/agentic-distillation-benchmark/checkpoints/base', 'steps': 0}


In [5]:
!python -m adbench.data.prepare --config configs/data.yaml

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
Loading glaiveai/glaive-function-calling-v2 ...
README.md: 100% 106/106 [00:00<00:00, 452kB/s]

glaive-function-calling-v2.json: downloading bytes:  15% 39.8M/271M [00:01<00:04, 56.2MB/s, 2.66MB/s  ]
glaive-function-calling-v2.json: downloading bytes:  21% 57.1M/271M [00:01<00:03, 66.2MB/s, 4.58MB/s  ]
glaive-function-calling-v2.json: downloading bytes:  38% 104M/271M [00:01<00:01, 87.5MB/s, 8.49MB/s  ] 
glaive-function-calling-v2.json: downloading bytes: 100% 104M/104M [00:02<00:00, 44.6MB/s, 9.55MB/s  ]
glaive-function-calling-v2.json: reconstructing file: 100%

## Condition 2 — sft_only (control)

Standard SFT on `data/splits/train.jsonl`, no teacher involved —
`resolve_training_config` forces `kd_weight=0` for this condition
regardless of `configs/experiment.yaml`'s `training.kd` block, so it can
never accidentally depend on the teacher.

It appears that the `train.jsonl` file, which is necessary for the `sft_only` condition, is missing. This file is generated by running `01_data_prep.ipynb`. Please ensure you have run that notebook first before proceeding.

In [8]:
sft_summary = train_condition("sft_only", experiment_config, models_config)
print(sft_summary)

==((====))==  Unsloth 2026.9.4: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

RuntimeError: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)

In [7]:
import torch
import re

file_path = "/content/agentic-distillation-benchmark/src/adbench/training/train.py"

with open(file_path, "r") as f:
    content = f.read()

# Ensure torch is imported at the very top if not already
if "import torch" not in content[:50]: # Check first 50 chars for import torch
    content = "import torch\n" + content

# Define device within the run_training_loop function
# This assumes the signature of run_training_loop is as seen in the traceback
if "device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')" not in content:
    content = content.replace(
        "def run_training_loop(student, teacher, examples, cfg, pad_token_id):",
        "def run_training_loop(student, teacher, examples, cfg, pad_token_id):\n" # Original function signature
        "    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n"
    )

# Patterns to find lines assigning input_ids, attention_mask, labels and add .to(device)
patterns_to_fix = [
    # This matches lines like: `        input_ids = batch["input_ids"]`
    (r"(^\s*input_ids\s*=\s*batch\[\"input_ids\"\])(?!\.to\(device\))", r"\1.to(device)"),
    # This matches lines like: `        attention_mask = batch["attention_mask"]`
    (r"(^\s*attention_mask\s*=\s*batch\[\"attention_mask\"\])(?!\.to\(device\))", r"\1.to(device)"),
    # This matches the specific line from the traceback: `        labels = torch.tensor(batch["labels"])`
    (r"(^\s*labels\s*=\s*torch\.tensor\(batch\[\"labels\"\]\))(?!\.to\(device\))", r"\1.to(device)"),
]

for old_pattern, new_pattern in patterns_to_fix:
    content = re.sub(old_pattern, new_pattern, content, flags=re.MULTILINE)

with open(file_path, "w") as f:
    f.write(content)

print("Patched 'src/adbench/training/train.py' to move tensors to the correct device.")


Patched 'src/adbench/training/train.py' to move tensors to the correct device.


## Condition 3 — distilled (treatment)

Combined KD (from the teacher) + SFT loss. Same data/LoRA/optimizer
settings as `sft_only` — only the loss function differs.

To try a named hyperparameter variation instead of the base `training.kd`
values (configs/experiment.yaml: `training.sweep`), pass `sweep_name=`:
```python
train_condition("distilled", experiment_config, models_config, sweep_name="higher_kd_temp")
```

In [ ]:
distilled_summary = train_condition("distilled", experiment_config, models_config)
print(distilled_summary)

## Sanity-check convergence before moving to evaluation

`sft_only` and `distilled` each wrote a per-step loss log to
`results/training_logs/<condition>.jsonl` (`sft_loss`/`kd_loss`/`total_loss`
logged separately — see `train.py::run_training_loop`). Plot them side by
side before spending eval time on a run that didn't converge.

In [ ]:
import matplotlib.pyplot as plt

from adbench.data.prepare import read_jsonl
from adbench.training.train import loss_log_path

sft_log = read_jsonl(loss_log_path("sft_only"))
distilled_log = read_jsonl(loss_log_path("distilled"))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot([r["step"] for r in sft_log], [r["sft_loss"] for r in sft_log], label="sft_only: sft_loss")
axes[0].plot([r["step"] for r in distilled_log], [r["sft_loss"] for r in distilled_log], label="distilled: sft_loss")
axes[0].set_title("SFT loss")
axes[0].set_xlabel("step")
axes[0].legend()

axes[1].plot([r["step"] for r in distilled_log], [r["kd_loss"] for r in distilled_log], label="distilled: kd_loss", color="darkorange")
axes[1].set_title("KD loss (distilled only)")
axes[1].set_xlabel("step")
axes[1].legend()
plt.tight_layout()
plt.show()